In [2]:
import numpy as np
import pandas as pd

In [3]:
trainingData = pd.read_csv("5x127x127_training_with_morphology.csv")
testingData = pd.read_csv("5x127x127_testing_with_morphology.csv")
validatingData = pd.read_csv("5x127x127_validation_with_morphology.csv")

In [4]:
print(f"{testingData.shape}, {validatingData.shape}, {testingData.shape}")

(40914, 84), (40914, 84), (40914, 84)


In [5]:
trainingData.shape

(204573, 84)

# Binary Classification

In [6]:
def classify_elliptical(df):
    n_i = df.get('i_sersic_index', pd.Series(np.nan, index=df.index)).astype(float)
    n_z = df.get('z_sersic_index', pd.Series(np.nan, index=df.index)).astype(float)
    n_y = df.get('y_sersic_index', pd.Series(np.nan, index=df.index)).astype(float)

    if ('i_minor_axis' in df.columns) and ('i_major_axis' in df.columns):
        ba_i = (df['i_minor_axis'].astype(float) / df['i_major_axis'].astype(float)).replace([np.inf, -np.inf], np.nan)
    else:
        ba_i = pd.Series(np.nan, index=df.index)

    peak_sb_i = df.get('i_peak_surface_brightness', pd.Series(np.nan, index=df.index)).astype(float)
    rhalf_i = df.get('i_half_light_radius', pd.Series(np.nan, index=df.index)).astype(float)

    gmag = df.get('g_cmodel_mag', pd.Series(np.nan, index=df.index)).astype(float)
    imag = df.get('i_cmodel_mag', pd.Series(np.nan, index=df.index)).astype(float)
    color_gi = gmag - imag

    N_STRONG = 2.6
    N_BASE = 2.5
    BA_EDGEON = 0.30

    finite_peak = peak_sb_i[np.isfinite(peak_sb_i)]
    sb_thresh = np.percentile(finite_peak, 30) if len(finite_peak) > 100 else 22.5
    compact_rhalf = 4.0

    label = (n_i >= N_BASE).astype(int)

    strong_core = (np.isfinite(peak_sb_i) & (peak_sb_i <= sb_thresh)) | (np.isfinite(rhalf_i) & (rhalf_i <= compact_rhalf))
    label = np.where((n_i >= N_STRONG) & strong_core, 1, label)

    red_support = ((n_z >= N_BASE) | (n_y >= N_BASE))
    border = (n_i >= 2.4) & (n_i < N_BASE)
    label = np.where(border & red_support, 1, label)

    edge_on = (ba_i < BA_EDGEON)
    label = np.where(edge_on & (n_i < N_STRONG) & ~strong_core, 0, label)

    very_red = (color_gi >= 1.2)
    very_blue = (color_gi <= 0.6)
    label = np.where(border & very_red, 1, label)
    label = np.where((n_i < N_BASE) & very_blue & ~strong_core, 0, label)

    return pd.Series(label.astype(int), index=df.index, name='elliptical')


In [7]:
#0 -> Non-elliptical
#1 -> Elliptical

In [8]:
binary_classif = classify_elliptical(trainingData)
binary_classif.value_counts()

elliptical
0    147431
1     57142
Name: count, dtype: int64

# Feature Engineering

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix

In [10]:
# Config
N_EARLY = 2.5
BA_MIN = 0.05
COMPACT_RHALF_PX = 4.0

CONTINUOUS_COLUMNS = [
    'n_i', 'n_z', 'n_y', 'ba_i', 'ellip_i', 'rhalf_i',
    'peak_sb_i', 'color_gi', 'color_ri'
]
BINARY_FLAG_COLUMNS = ['nz_agree_25', 'ny_agree_25', 'is_compact_px']

In [11]:
def _safe_div(a, b):
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    out = pd.Series(np.nan, index=a.index)
    mask = (b != 0) & np.isfinite(a) & np.isfinite(b)
    out.loc[mask] = a.loc[mask] / b.loc[mask]
    return out

def build_features(df):
    feats = pd.DataFrame(index=df.index)
    feats['object_id'] = df['object_id'] if 'object_id' in df.columns else np.arange(len(df))

    feats['n_i'] = df.get('i_sersic_index', pd.Series(np.nan, index=df.index))
    feats['n_z'] = df.get('z_sersic_index', pd.Series(np.nan, index=df.index))
    feats['n_y'] = df.get('y_sersic_index', pd.Series(np.nan, index=df.index))

    if ('i_minor_axis' in df.columns) and ('i_major_axis' in df.columns):
        ba_i = _safe_div(df['i_minor_axis'], df['i_major_axis']).clip(lower=BA_MIN, upper=1.0)
    else:
        ba_i = pd.Series(np.nan, index=df.index)
    feats['ba_i'] = ba_i

    feats['ellip_i'] = df.get('i_ellipticity', pd.Series(np.nan, index=df.index))
    feats['rhalf_i'] = df.get('i_half_light_radius', pd.Series(np.nan, index=df.index))
    feats['peak_sb_i'] = df.get('i_peak_surface_brightness', pd.Series(np.nan, index=df.index))

    gmag = df.get('g_cmodel_mag', pd.Series(np.nan, index=df.index))
    rmag = df.get('r_cmodel_mag', pd.Series(np.nan, index=df.index))
    imag = df.get('i_cmodel_mag', pd.Series(np.nan, index=df.index))
    feats['color_gi'] = pd.to_numeric(gmag, errors='coerce') - pd.to_numeric(imag, errors='coerce')
    feats['color_ri'] = pd.to_numeric(rmag, errors='coerce') - pd.to_numeric(imag, errors='coerce')

    feats['nz_agree_25'] = (pd.to_numeric(feats['n_z'], errors='coerce') >= N_EARLY).astype(float)
    feats['ny_agree_25'] = (pd.to_numeric(feats['n_y'], errors='coerce') >= N_EARLY).astype(float)
    feats['is_compact_px'] = (pd.to_numeric(feats['rhalf_i'], errors='coerce') <= COMPACT_RHALF_PX).astype(float)

    keep_cols = ['object_id'] + CONTINUOUS_COLUMNS + BINARY_FLAG_COLUMNS
    for c in keep_cols:
        if c not in feats.columns:
            feats[c] = np.nan
    feats = feats[keep_cols]
    return feats


In [12]:
# Assume trainingData is already loaded as in your script
y_all = classify_elliptical(trainingData)          # labels from raw trainingData
X_all = build_features(trainingData)               # features from raw trainingData

# Keep object_id aside (not a feature)
X_all_nofe = X_all.drop(columns=['object_id'])


In [13]:
X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(
    X_all_nofe, y_all, test_size=0.25, random_state=42, stratify=y_all
)


In [14]:
def fit_preprocessors_on_train(X_train):
    cont_cols = [c for c in CONTINUOUS_COLUMNS if c in X_train.columns]
    bin_cols  = [c for c in BINARY_FLAG_COLUMNS if c in X_train.columns]

    X_train = X_train.copy()
    for c in cont_cols:
        X_train[c] = pd.to_numeric(X_train[c], errors='coerce')

    cont_imputer = None
    scaler = None
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy='median')
        X_train[cont_cols] = cont_imputer.fit_transform(X_train[cont_cols])

        scaler = StandardScaler()
        X_train[cont_cols] = scaler.fit_transform(X_train[cont_cols])

    bin_imputer = None
    if len(bin_cols) > 0:
        bin_imputer = SimpleImputer(strategy='most_frequent')
        X_train[bin_cols] = bin_imputer.fit_transform(X_train[bin_cols])

    transf = {
        'cont_imputer': cont_imputer,
        'scaler': scaler,
        'bin_imputer': bin_imputer,
        'cont_cols': cont_cols,
        'bin_cols': bin_cols
    }
    return X_train, transf

def transform_with_preprocessors(X, transf):
    X = X.copy()
    cont_cols = transf['cont_cols']
    bin_cols  = transf['bin_cols']

    for c in cont_cols:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    if transf['cont_imputer'] is not None and len(cont_cols) > 0:
        X[cont_cols] = transf['cont_imputer'].transform(X[cont_cols])

    if transf['scaler'] is not None and len(cont_cols) > 0:
        X[cont_cols] = transf['scaler'].transform(X[cont_cols])

    if transf['bin_imputer'] is not None and len(bin_cols) > 0:
        X[bin_cols] = transf['bin_imputer'].transform(X[bin_cols])

    return X

# Fit on train
X_tr, transf = fit_preprocessors_on_train(X_raw_tr)
# Transform test with same transf
X_te = transform_with_preprocessors(X_raw_te, transf)


### Random Forest Classifier

In [15]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_tr, y_tr)

USE_CALIBRATION = True
if USE_CALIBRATION:
    rf_cal = CalibratedClassifierCV(rf, method='isotonic', cv=3)
    rf_cal.fit(X_tr, y_tr)
    model_final = rf_cal
else:
    model_final = rf

y_pred = model_final.predict(X_te)
try:
    y_proba = model_final.predict_proba(X_te)[:, 1]
except Exception:
    y_proba = None

print("=== Random Forest Evaluation ===")
print(classification_report(y_te, y_pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_te, y_pred))


=== Random Forest Evaluation ===
              precision    recall  f1-score   support

           0     1.0000    0.9999    0.9999     36858
           1     0.9997    1.0000    0.9999     14286

    accuracy                         0.9999     51144
   macro avg     0.9999    0.9999    0.9999     51144
weighted avg     0.9999    0.9999    0.9999     51144

Confusion matrix:
[[36854     4]
 [    0 14286]]


In [16]:
importances = pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values(ascending=False)
print("\nTop feature importances:")
print(importances.head(15))

results = pd.DataFrame({
    'elliptical_true': y_te.values,
    'elliptical_pred': y_pred
}, index=X_te.index)
if y_proba is not None:
    results['proba'] = y_proba
print("\nResults preview:")
print(results.head())



Top feature importances:
n_i              0.685452
n_z              0.133485
n_y              0.076304
nz_agree_25      0.029570
peak_sb_i        0.026777
ny_agree_25      0.015901
color_gi         0.012802
rhalf_i          0.011918
color_ri         0.003498
ba_i             0.001712
ellip_i          0.001297
is_compact_px    0.001283
dtype: float64

Results preview:
        elliptical_true  elliptical_pred     proba
32994                 1                1  0.998984
22056                 0                0  0.000000
127751                0                0  0.000000
143332                1                1  1.000000
59214                 0                0  0.000000


In [17]:
feats_scaled_full = transform_with_preprocessors(X_all_nofe, transf)
new_table = feats_scaled_full.copy()
new_table['elliptical'] = y_all.astype(int).values
# Optionally add object_id back if you want alignment
new_table = pd.concat([X_all[['object_id']].reset_index(drop=True),
                       new_table.reset_index(drop=True)], axis=1)

print(new_table.shape)
print(new_table.columns.tolist())


(204573, 14)
['object_id', 'n_i', 'n_z', 'n_y', 'ba_i', 'ellip_i', 'rhalf_i', 'peak_sb_i', 'color_gi', 'color_ri', 'nz_agree_25', 'ny_agree_25', 'is_compact_px', 'elliptical']


### XGB Boosting

In [18]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

# Try to use XGBoost if installed; else fallback to sklearn GB
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# 1) Build the boosting model
if HAVE_XGB:
    gb = XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    model_name = "XGBoost"
else:
    gb = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
    model_name = "GradientBoosting"

# 2) Fit on train
gb.fit(X_tr, y_tr)

# Optional: probability calibration
USE_CALIBRATION = True
if USE_CALIBRATION:
    gb_cal = CalibratedClassifierCV(gb, method='isotonic', cv=3)
    gb_cal.fit(X_tr, y_tr)
    gb_final = gb_cal
else:
    gb_final = gb

# 3) Evaluate
y_pred_gb = gb_final.predict(X_te)
try:
    y_proba_gb = gb_final.predict_proba(X_te)[:, 1]
except Exception:
    y_proba_gb = None

print(f"=== {model_name} Evaluation ===")
print(classification_report(y_te, y_pred_gb, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_te, y_pred_gb))

# 4) Feature importances (tree-based models only)
def get_importances(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
    # For calibrated models, use the base estimator if available
    if hasattr(model, 'base_estimator') and hasattr(model.base_estimator, 'feature_importances_'):
        return pd.Series(model.base_estimator.feature_importances_, index=feature_names).sort_values(ascending=False)
    return None

gb_importances = get_importances(gb, X_tr.columns)
if gb_importances is not None:
    print("\nTop feature importances:")
    print(gb_importances.head(15))
else:
    print("\nNo feature_importances_ available for this model.")

# 5) Results table for quick inspection
gb_results = pd.DataFrame({
    'elliptical_true': y_te.values,
    'elliptical_pred': y_pred_gb
}, index=X_te.index)
if y_proba_gb is not None:
    gb_results['proba'] = y_proba_gb

print("\nResults preview:")
print(gb_results.head())

# Optional: save
# gb_results.to_csv('gb_results.csv', index=False)
# if gb_importances is not None:
#     gb_importances.to_csv('gb_feature_importances.csv')


=== XGBoost Evaluation ===
              precision    recall  f1-score   support

           0     0.9996    0.9998    0.9997     36858
           1     0.9994    0.9989    0.9992     14286

    accuracy                         0.9995     51144
   macro avg     0.9995    0.9993    0.9994     51144
weighted avg     0.9995    0.9995    0.9995     51144

Confusion matrix:
[[36850     8]
 [   16 14270]]

Top feature importances:
n_i              0.866433
n_z              0.071542
n_y              0.011968
nz_agree_25      0.010391
is_compact_px    0.009877
color_gi         0.008203
rhalf_i          0.007239
ny_agree_25      0.006210
peak_sb_i        0.004298
color_ri         0.002429
ba_i             0.000741
ellip_i          0.000668
dtype: float32

Results preview:
        elliptical_true  elliptical_pred  proba
32994                 1                1    1.0
22056                 0                0    0.0
127751                0                0    0.0
143332                1           

# Testing & Validation

In [19]:
# Labels from your rule (same function used for training labels)
y_val  = classify_elliptical(validatingData).astype(int)
y_test = classify_elliptical(testingData).astype(int)

# Raw feature engineering (no scaling yet)
X_val_raw  = build_features(validatingData)
X_test_raw = build_features(testingData)

# Drop ID for model input; keep object_id separately if you want later
X_val_nofe  = X_val_raw.drop(columns=['object_id'])
X_test_nofe = X_test_raw.drop(columns=['object_id'])


In [20]:
X_val  = transform_with_preprocessors(X_val_nofe, transf)
X_test = transform_with_preprocessors(X_test_nofe, transf)

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, X, y, name="Model"):
    y_pred = model.predict(X)
    try:
        y_proba = model.predict_proba(X)[:, 1]
    except Exception:
        y_proba = None
    print(f"\n=== {name} Evaluation ===")
    print(classification_report(y, y_pred, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(y, y_pred))
    return y_pred, y_proba

# Replace the model variable names below to match what you actually have in memory:
# Example: model_final_rf and model_final_gb; or just model_final for RF and model_final_gb for boosting.

# Validation evaluations
rf_val_pred,  rf_val_proba  = evaluate_model(model_final,  X_val,  y_val,  name="Random Forest (Validation)")
gb_val_pred,  gb_val_proba  = evaluate_model(gb_final,  X_val,  y_val,  name="XGBoost/GB (Validation)")

# Test evaluations
rf_test_pred, rf_test_proba = evaluate_model(model_final,  X_test, y_test, name="Random Forest (Test)")
gb_test_pred, gb_test_proba = evaluate_model(gb_final,  X_test, y_test, name="XGBoost/GB (Test)")



=== Random Forest (Validation) Evaluation ===
              precision    recall  f1-score   support

           0     1.0000    0.9999    0.9999     29276
           1     0.9997    1.0000    0.9998     11638

    accuracy                         0.9999     40914
   macro avg     0.9998    0.9999    0.9999     40914
weighted avg     0.9999    0.9999    0.9999     40914

Confusion matrix:
[[29272     4]
 [    0 11638]]

=== XGBoost/GB (Validation) Evaluation ===
              precision    recall  f1-score   support

           0     0.9995    0.9998    0.9997     29276
           1     0.9996    0.9987    0.9991     11638

    accuracy                         0.9995     40914
   macro avg     0.9995    0.9993    0.9994     40914
weighted avg     0.9995    0.9995    0.9995     40914

Confusion matrix:
[[29271     5]
 [   15 11623]]

=== Random Forest (Test) Evaluation ===
              precision    recall  f1-score   support

           0     0.9999    1.0000    0.9999     29400
       

In [22]:
val_table = X_val.copy()
val_table['elliptical'] = y_val.values
val_table = pd.concat([X_val_raw[['object_id']].reset_index(drop=True),
                       val_table.reset_index(drop=True)], axis=1)

test_table = X_test.copy()
test_table['elliptical'] = y_test.values
test_table = pd.concat([X_test_raw[['object_id']].reset_index(drop=True),
                        test_table.reset_index(drop=True)], axis=1)

print(val_table.shape, test_table.shape)


(40914, 14) (40914, 14)
